In [1]:
import json
import re
from pathlib import Path
import pandas as pd
from rapidfuzz import process, fuzz
import spacy

In [2]:
with open('../gweb_export_old.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

OLD_PRODUCTS_DATABASE = data['products']

In [13]:
nlp = spacy.load("pt_core_news_sm")

synonyms = {
    'ads': 'adesivo',
    'pf': 'parafuso',
    'transm': 'transmissao',
    'c': 'cilindro',
    'amort': 'amortecedor',
    'patins': 'pastilha',
    'sup': 'suporte',
    'valv': 'valvula',
    'igin': 'iguinição',
    'capac': 'capacete',
    'inter': 'interruptor'
}

# Criamos um "Set" de preposições (é mais rápido que uma lista para o Python buscar)
prepositions = {'de', 'da', 'do', 'das', 'dos', 'com', 'para', 'em', 'no', 'na', 'por', 'c/'}

words_ignore = {
    'diant', 'dianteiro', 'dianteira', 'tras', 'traseiro', 'traseira',
    'tit', 'cg', 'es', 'ks', 'mix', 'esd'
}

ncm_rules = {
    'capacete': '65061000'
}

In [4]:
def extract_first_tokens(description, n_tokens=2):
    if not description:
        return []

    doc = nlp(description)
    extracted_tokens = [
        token.text for token in doc 
        if not token.is_space 
        and token.text not in prepositions
        and token.text not in words_ignore
    ][:n_tokens]

    return extracted_tokens

In [7]:
def normalize_product_name(text, n_tokens=2):
    if not text:
        return ''

    text = text.lower().strip()
    
    # 1. REMOVE anos de fabricação (ex: 2014/2015, 14-15)
    text = re.sub(r'\b\d{2,4}\s*[-/]\s*\d{2,4}\b', ' ', text)

    # 2. REMOVE modelos (ex: fan 125)
    text = re.sub(r'\b[a-z]+\s*\d+\b', ' ', text)
    
    # 3. REMOVE barras e o que estiver grudado nela (ex: /abs)
    text = re.sub(r'/[a-z0-9]*', ' ', text)
    
    # 4. REMOVE absolutamente qualquer número (isolado ou não)
    text = re.sub(r'\d+', ' ', text)
    
    # 5. NOVO: REMOVE caracteres especiais específicos (/, -, +)
    # A barra invertida antes do - e do + é para o Python entender que é o texto literal
    text = re.sub(r'[/\-\+\.]', ' ', text)
    
    # 6. Limpeza de manutenção: remove espaços duplos
    text = re.sub(r'\s+', ' ', text).strip()
    
    tokenized_tokens = extract_first_tokens(text, n_tokens)
    normalized_tokens = [synonyms.get(token, token) for token in tokenized_tokens]

    return ' '.join(normalized_tokens)

In [8]:
references_to_old_products = {
    normalize_product_name(p['name']): p['details'][0]['ncm_code']
    for p in OLD_PRODUCTS_DATABASE
}
old_product_names = list(references_to_old_products.keys())

In [14]:
def get_ncm_by_rule(product):
    for keyword, ncm in ncm_rules.items():
        if keyword in product:
            return ncm
    return ''

In [15]:
def get_match_result(product):
    if not product:
        return None

    clean_product_name = normalize_product_name(product)
    
    get_ncm_by_rule(clean_product_name)
    match = process.extractOne(
        clean_product_name,
        old_product_names,
        scorer=fuzz.token_set_ratio,
        score_cutoff=66
    )

    if match is None:
        return ''

    product_name, score, *_ = match
    
    return {
        'combined_product': clean_product_name,
        'product_match': product_name,
        'corresponding_ncm': str(references_to_old_products[product_name]),
        'score': score
    }

In [10]:
df = pd.read_excel('./clean_products_database.xlsx', usecols=[0, 1, 2, 3, 4])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3022 entries, 0 to 3021
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          3022 non-null   int64  
 1   CÓD. BARRA  2857 non-null   str    
 2   DESCRIÇÃO   3022 non-null   str    
 3   QUANTIDADE  3022 non-null   int64  
 4   PREÇO       3022 non-null   float64
dtypes: float64(1), int64(2), str(2)
memory usage: 118.2 KB


In [16]:
results = []
for p in df['DESCRIÇÃO']:
    result = get_match_result(str(p))
    print(f'{result}\n')
    results.append(result)
df['NCM'] = [res['corresponding_ncm'] if res else None for res in results]

df.info()

{'combined_product': 'reparo freio', 'product_match': 'reparo freio', 'corresponding_ncm': '87141000', 'score': 100.0}

{'combined_product': 'reparo freio', 'product_match': 'reparo freio', 'corresponding_ncm': '87141000', 'score': 100.0}

{'combined_product': 'reparo freio', 'product_match': 'reparo freio', 'corresponding_ncm': '87141000', 'score': 100.0}

{'combined_product': 'reparo freio', 'product_match': 'reparo freio', 'corresponding_ncm': '87141000', 'score': 100.0}

{'combined_product': 'filtro oleo', 'product_match': 'filtro', 'corresponding_ncm': '84213100', 'score': 100.0}

{'combined_product': 'filtro oleo', 'product_match': 'filtro', 'corresponding_ncm': '84213100', 'score': 100.0}

{'combined_product': 'filtro oleo', 'product_match': 'filtro', 'corresponding_ncm': '84213100', 'score': 100.0}

{'combined_product': 'parafuso arvore', 'product_match': 'arvore', 'corresponding_ncm': '87141000', 'score': 100.0}

{'combined_product': 'sensor oxigenio', 'product_match': 'sensor

In [17]:
df.to_excel('../conversion/clean_products_conversion.xlsx', index=False)